# H15: Conditioning-based probe of Muon in an alternating diagonal/matrix toy model

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/H15_NORMALIZATION_GAUGE_DUALITY/run_experiment.py`

This notebook is the analysis/report companion to the paired script and intentionally **uses the script's returned results** rather than re-implementing the training loop.

## What this pair actually studies

- a **toy alternating-layer model** with 4 layers: diagonal → matrix → diagonal → matrix
- the same default H15 setup as the script: `DIM=32`, `NUM_STEPS=500`, `NUM_SEEDS=5`, synthetic Gaussian data, momentum SGD vs. Muon-style updates
- two descriptive observables:
  - training loss
  - per-layer weight condition number $\kappa(W)$
- diagonal-vs-matrix layer comparisons treated as a **heuristic conditioning probe**

## What this pair does **not** claim

- matrix-layer $\kappa(W)$ is **not** treated as a direct gauge-coordinate or gauge-drift observable
- matrix-vs-diagonal $\kappa$ ratios are **not** presented as a clean causal decomposition of a gauge-specific contribution
- this notebook does **not** provide formal significance testing or a proof of gauge duality

The correct reading of H15 in this first completion pass is therefore: **a conditioning-based toy probe whose default run shows strong loss improvement for Muon, but not the originally intended matrix-layer conditioning advantage story.**


## 1. Setup and notebook-safe import handling

The notebook avoids `__file__` and searches upward from the current working directory for the paired script. This makes the import logic explicit and notebook-safe.


In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")


def md(text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(text))
    except Exception:
        print(text)


CANDIDATE_REL = Path("experiments/Tier_1_Core_Mechanism_Tests/H15_NORMALIZATION_GAUGE_DUALITY/run_experiment.py")
search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
SCRIPT_PATH = None
REPO_ROOT = None

for base in search_roots:
    candidate = (base / CANDIDATE_REL).resolve()
    if candidate.exists():
        SCRIPT_PATH = candidate
        REPO_ROOT = base.resolve()
        break

if SCRIPT_PATH is None:
    raise FileNotFoundError(
        f"Could not locate {CANDIDATE_REL} by searching upward from {Path.cwd().resolve()}"
    )

EXPERIMENT_DIR = SCRIPT_PATH.parent
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import run_experiment as h15

assert Path(h15.MODULE_PATH).resolve() == SCRIPT_PATH

md(
    f"""
**Imported script:** `{SCRIPT_PATH.relative_to(REPO_ROOT)}`  
**Notebook counterpart:** `{Path(h15.NOTEBOOK_PATH).relative_to(REPO_ROOT)}`

This notebook is intentionally `__file__`-free and reuses the script module directly.
"""
)
print(f"numpy={np.__version__}")
print(f"matplotlib={matplotlib.__version__}")


## 2. Reproducibility, runtime, and experiment identity

The next cell runs the paired script's `run_experiment()` function directly, records wall-clock time for this session, and logs the exact default configuration and measurement caveats preserved by the script.


In [ ]:
start_time = time.perf_counter()
results = h15.run_experiment(verbose=False)
elapsed_s = time.perf_counter() - start_time

config = results["config"]
summary = results["summary"]
aggregates = results["aggregates"]
paired_stats = aggregates["paired_statistics"]
raw = results["raw"]
measurement_notes = results["measurement_notes"]
layer_names = results["layer_names"]
layer_types = results["layer_types"]
seeds = results["seeds"]

md(
    f"""
### Reproducibility log
- **Experiment ID:** `{results['identity']['experiment_id']}`
- **Script:** `{Path(results['paths']['script']).resolve().relative_to(REPO_ROOT)}`
- **Notebook counterpart:** `{Path(results['paths']['notebook']).resolve().relative_to(REPO_ROOT)}`
- **Seeds:** `{seeds}`
- **Configuration:** `dim={config['dim']}`, `steps={config['num_steps']}`, `samples={config['num_samples']}`, `lr_sgd={config['lr_sgd']}`, `lr_muon={config['lr_muon']}`, `momentum={config['momentum']}`, `ns_iters={config['ns_iters']}`
- **Execution time in this session:** `{elapsed_s:.2f} s`
- **Last recorded step:** `{summary['last_recorded_step']}`
"""
)

print("Measurement timing note:")
print(" ", measurement_notes["loss_and_kappa_timing"])
print("Conditioning-scope note:")
print(" ", measurement_notes["scope"])
print("Matrix-vs-diagonal ratio caveat:")
print(" ", measurement_notes["ratio_caveat"])


## 3. Helper utilities for tables and interpretation

These are lightweight notebook-side helpers only. The core experiment logic remains in the script.


In [ ]:
def render_markdown_table(headers, rows):
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    md("\n".join(lines))


## 4. Mean loss trajectory

The first figure checks whether Muon's optimization dynamics produce a meaningful loss advantage in the default toy setup. Shaded bands show **±1 standard deviation across seeds**.

Because the same seeds are run under both optimizers, the notebook also reports a **paired final-loss summary** derived from the seedwise differences and ratios. These summaries improve evidence presentation, but they are still **descriptive rather than inferential**.


In [ ]:
steps = np.arange(config["num_steps"])
loss_gap_stats = paired_stats["final_loss_gap_sgd_minus_muon"]
loss_ratio_stats = paired_stats["final_loss_ratio_sgd_over_muon"]

fig, ax = plt.subplots(figsize=(8.5, 5.0))
for label, data, color in [
    ("SGD", raw["sgd_losses"], "tab:orange"),
    ("Muon", raw["muon_losses"], "tab:blue"),
]:
    mean = data.mean(axis=0)
    std = data.std(axis=0)
    ax.plot(steps, mean, label=f"{label} mean", color=color, linewidth=2)
    ax.fill_between(steps, mean - std, mean + std, color=color, alpha=0.2, label=f"{label} ±1 std")

ax.set_title("Mean loss trajectory across seeds")
ax.set_xlabel("training step (recorded pre-update)")
ax.set_ylabel("loss")
ax.legend(ncols=2, fontsize=9)
plt.tight_layout()
plt.show()

print(
    f"Last recorded mean loss: SGD={summary['final_sgd_loss_mean']:.6f} ± {summary['final_sgd_loss_std']:.6f}, "
    f"Muon={summary['final_muon_loss_mean']:.6f} ± {summary['final_muon_loss_std']:.6f}"
)
print(f"SGD/Muon loss ratio at the last recorded step: {summary['loss_ratio_sgd_over_muon']:.3f}x")

render_markdown_table(
    ["Paired final-loss statistic", "Mean ± std", "SEM", "Median", "Range"],
    [
        [
            "Seedwise loss gap (SGD − Muon)",
            f"{loss_gap_stats['mean']:.6f} ± {loss_gap_stats['std']:.6f}",
            f"{loss_gap_stats['sem']:.6f}",
            f"{loss_gap_stats['median']:.6f}",
            f"[{loss_gap_stats['min']:.6f}, {loss_gap_stats['max']:.6f}]",
        ],
        [
            "Seedwise loss ratio (SGD / Muon)",
            f"{loss_ratio_stats['mean']:.3f}x ± {loss_ratio_stats['std']:.3f}x",
            f"{loss_ratio_stats['sem']:.3f}x",
            f"{loss_ratio_stats['median']:.3f}x",
            f"[{loss_ratio_stats['min']:.3f}x, {loss_ratio_stats['max']:.3f}x]",
        ],
    ],
)


In [ ]:
if summary["final_muon_loss_mean"] < summary["final_sgd_loss_mean"]:
    md(
        f"""
**Interpretation.** Muon shows a strong optimization advantage in the default H15 toy setup.  
The last recorded mean loss decreases from `{summary['final_sgd_loss_mean']:.4f}` under SGD to `{summary['final_muon_loss_mean']:.4f}` under Muon, corresponding to an SGD/Muon ratio of `{summary['loss_ratio_sgd_over_muon']:.2f}x`.

The paired seedwise final-loss gap `(SGD − Muon)` is also positive on average: `{summary['paired_final_loss_gap_mean']:.4f} ± {summary['paired_final_loss_gap_std']:.4f}` with SEM `{summary['paired_final_loss_gap_sem']:.4f}`. This is stronger descriptive evidence than only comparing unpaired means, but it is still not presented as a formal significance test.
"""
    )
else:
    md(
        """
**Interpretation.** In this run Muon does not beat SGD on the last recorded mean loss, so any stronger narrative would need to be reconsidered.
"""
    )


## 5. Conditioning trajectories by layer and by layer type

The next figures show the actual observable used in H15: **weight condition number**.

- Diagonal layers use $\kappa(D) = \max |d_i| / \min |d_i|$.
- Matrix layers use the singular-value condition number $\kappa(W) = \sigma_{\max}/\sigma_{\min}$.

A log scale is used because condition numbers can span orders of magnitude. This is still a **conditioning diagnostic**, not a direct gauge-coordinate measurement.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for li, ax in enumerate(axes.flat):
    sgd_mean = aggregates["sgd_kappas_mean"][:, li]
    sgd_std = aggregates["sgd_kappas_std"][:, li]
    muon_mean = aggregates["muon_kappas_mean"][:, li]
    muon_std = aggregates["muon_kappas_std"][:, li]

    ax.plot(steps, sgd_mean, color="tab:orange", linewidth=2, label="SGD")
    ax.fill_between(
        steps,
        np.maximum(sgd_mean - sgd_std, 1e-12),
        sgd_mean + sgd_std,
        color="tab:orange",
        alpha=0.15,
    )
    ax.plot(steps, muon_mean, color="tab:blue", linewidth=2, label="Muon")
    ax.fill_between(
        steps,
        np.maximum(muon_mean - muon_std, 1e-12),
        muon_mean + muon_std,
        color="tab:blue",
        alpha=0.15,
    )

    ax.set_yscale("log")
    ax.set_title(f"{layer_names[li]} — {layer_types[li]}")
    ax.set_xlabel("training step")
    ax.set_ylabel(r"condition number $\kappa(W)$")
    if li == 0:
        ax.legend()

fig.suptitle("Per-layer conditioning trajectories (mean ± 1 std across seeds)", y=1.02)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)
for ax, key, title in zip(
    axes,
    ["diag", "matrix"],
    ["Diagonal layers (type-averaged)", "Matrix layers (type-averaged)"],
):
    row = aggregates["per_type_final"][key]
    ax.plot(steps, row["sgd_trajectory_mean"], color="tab:orange", linewidth=2, label="SGD")
    ax.fill_between(
        steps,
        np.maximum(row["sgd_trajectory_mean"] - row["sgd_trajectory_std"], 1e-12),
        row["sgd_trajectory_mean"] + row["sgd_trajectory_std"],
        color="tab:orange",
        alpha=0.15,
    )
    ax.plot(steps, row["muon_trajectory_mean"], color="tab:blue", linewidth=2, label="Muon")
    ax.fill_between(
        steps,
        np.maximum(row["muon_trajectory_mean"] - row["muon_trajectory_std"], 1e-12),
        row["muon_trajectory_mean"] + row["muon_trajectory_std"],
        color="tab:blue",
        alpha=0.15,
    )
    ax.set_yscale("log")
    ax.set_title(title)
    ax.set_xlabel("training step")
    ax.set_ylabel(r"mean condition number $\kappa(W)$")
    ax.legend()

fig.suptitle("Type-averaged conditioning trajectories", y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
if summary["avg_matrix_ratio"] > summary["avg_diag_ratio"]:
    md(
        f"""
**Interpretation.** Matrix layers end with the larger conditioning improvement on this metric: diagonal layers show a mean layerwise SGD/Muon ratio of `{summary['avg_diag_ratio']:.2f}x`, while matrix layers show `{summary['avg_matrix_ratio']:.2f}x`.

That pattern would still need to be interpreted cautiously, because the observable is a conditioning metric rather than a direct gauge measurement.
"""
    )
else:
    md(
        f"""
**Interpretation.** The default H15 run does **not** support the originally intended matrix-layer conditioning advantage story.  
At the last recorded step, the mean layerwise SGD/Muon ratio is `{summary['avg_diag_ratio']:.2f}x` for diagonal layers but only `{summary['avg_matrix_ratio']:.2f}x` for matrix layers, giving a matrix-over-diagonal ratio of `{summary['matrix_over_diag_ratio']:.3f}x`.
"""
    )


## 6. Compact final-summary tables

The tables below summarize the last recorded step of the run. The first table is layer-by-layer and includes the spread of the **per-seed** SGD/Muon conditioning ratio for each layer. The second aggregates diagonal and matrix layers separately and now also exposes the corresponding **SEM** values as lightweight uncertainty summaries.


In [ ]:
layer_rows = []
for row in aggregates["per_layer_final"]:
    layer_rows.append([
        row["layer_name"],
        row["layer_type"],
        f"{row['sgd_mean']:.2f} ± {row['sgd_std']:.2f}",
        f"{row['muon_mean']:.2f} ± {row['muon_std']:.2f}",
        f"{row['ratio_of_means']:.2f}x",
        f"{row['per_seed_ratio_mean']:.2f} ± {row['per_seed_ratio_std']:.2f}",
        f"{row['per_seed_ratio_sem']:.2f}",
    ])

render_markdown_table(
    [
        "Layer",
        "Type",
        "SGD final $\kappa$",
        "Muon final $\kappa$",
        "Ratio of means",
        "Per-seed ratio mean ± std",
        "Ratio SEM",
    ],
    layer_rows,
)

type_rows = []
for key in ["diag", "matrix"]:
    row = aggregates["per_type_final"][key]
    type_rows.append([
        key,
        f"{row['sgd_mean']:.2f} ± {row['sgd_std']:.2f}",
        f"{row['muon_mean']:.2f} ± {row['muon_std']:.2f}",
        f"{row['layer_ratio_mean']:.2f}x",
        f"{row['per_seed_ratio_mean']:.2f} ± {row['per_seed_ratio_std']:.2f}",
        f"{row['per_seed_ratio_sem']:.2f}",
    ])

render_markdown_table(
    [
        "Layer type",
        "SGD type-avg final $\kappa$",
        "Muon type-avg final $\kappa$",
        "Mean of layerwise ratios",
        "Mean ± std of per-seed type ratios",
        "Type-ratio SEM",
    ],
    type_rows,
)

divergence_rows = []
for optimizer, flags in aggregates["divergence"].items():
    divergence_rows.append([optimizer, sum(bool(x) for x in flags), len(flags), flags])

render_markdown_table(
    ["Optimizer", "Diverged seeds", "Total seeds", "Per-seed flags"],
    divergence_rows,
)

render_markdown_table(
    ["Additional summary metric", "Value"],
    [
        ["Matrix-over-diagonal improvement ratio (heuristic only)", f"{summary['matrix_over_diag_ratio']:.3f}x"],
        ["Last recorded SGD mean loss", f"{summary['final_sgd_loss_mean']:.6f} ± {summary['final_sgd_loss_std']:.6f}"],
        ["Last recorded Muon mean loss", f"{summary['final_muon_loss_mean']:.6f} ± {summary['final_muon_loss_std']:.6f}"],
        ["Paired final loss gap (SGD − Muon)", f"{summary['paired_final_loss_gap_mean']:.6f} ± {summary['paired_final_loss_gap_std']:.6f} (SEM {summary['paired_final_loss_gap_sem']:.6f})"],
        ["Paired final loss ratio (SGD / Muon)", f"{summary['paired_final_loss_ratio_mean']:.3f}x ± {summary['paired_final_loss_ratio_std']:.3f}x (SEM {summary['paired_final_loss_ratio_sem']:.3f}x)"],
    ],
)


## 7. H1–H4 heuristic verdicts

These checks are preserved from the original H15 setup, but the framing is narrowed: they are **descriptive heuristic checks**, not formal inferential tests.


In [ ]:
h_rows = []
for key in ["H1", "H2", "H3", "H4"]:
    item = results["hypotheses"][key]
    if key == "H4":
        obs = item["observed_value"]
        observed = (
            f"SGD={obs['sgd_mean']:.4f}, Muon={obs['muon_mean']:.4f}, "
            f"SGD/Muon={obs['sgd_over_muon']:.2f}x"
        )
    elif key == "H3":
        observed = f"matrix/diag improvement ratio = {item['observed_value']:.3f}x"
    else:
        observed = f"{item['observed_value']:.3f}x"

    h_rows.append([
        key,
        item["description"],
        item["criterion"],
        observed,
        "PASS" if item["passed"] else "FAIL",
    ])

render_markdown_table(
    ["Hypothesis", "Meaning", "Criterion", "Observed", "Verdict"],
    h_rows,
)

md(f"**Total heuristic passes:** {summary['total_pass']}/4")


## 8. Calibrated conclusion

The goal of this cell is to state exactly what the current run supports and what it does not.


In [ ]:
md(
    f"""
### Final conclusion

- **Strong supported claim:** Muon substantially improves optimization in this toy setup, reducing the last recorded mean loss from `{summary['final_sgd_loss_mean']:.4f}` to `{summary['final_muon_loss_mean']:.4f}`.
- **Second-pass evidence upgrade:** on the same seeds, the paired final-loss gap `(SGD − Muon)` averages `{summary['paired_final_loss_gap_mean']:.4f} ± {summary['paired_final_loss_gap_std']:.4f}` with SEM `{summary['paired_final_loss_gap_sem']:.4f}`, and the paired final-loss ratio averages `{summary['paired_final_loss_ratio_mean']:.2f}x`.
- **Important negative result:** the current default run does **not** support the originally intended matrix-layer conditioning advantage story; diagonal layers show the larger final conditioning gain on the present metric.
- **Interpretation boundary:** this pair studies a conditioning-based toy probe in an alternating diagonal/matrix architecture. Matrix-layer $\kappa(W)$ is a useful spectral conditioning diagnostic here, but it is **not** a direct gauge observable.
- **Method caveats:** only `n=5` seeds are used, all uncertainty summaries here are descriptive only, no formal significance test is performed, and the stored histories are measured **pre-update** at each step to preserve the original experiment semantics.

Taken together, the serious second-pass reading is unchanged: **Muon clearly helps loss, but this H15 configuration does not currently validate a strong matrix-layer-over-diagonal conditioning advantage claim.**
"""
)
